# 🔗 Markov Chain Text Generation
> **ProDigy Infotech · Machine Learning Internship · Task 03**

Implement a text generation algorithm using Markov chains — a statistical model that predicts the next word (or character) based on the previous one(s).

## 📦 Install Dependencies

In [ ]:
!pip install markovify -q

## 📚 Imports

In [ ]:
import random
import re
import json
import markovify
from collections import defaultdict
from typing import Optional

print('✅ All imports successful')

## 📄 Load Training Data

In [ ]:
with open('train_data.txt', 'r', encoding='utf-8') as f:
    TEXT = f.read()

print(f'Loaded {len(TEXT)} characters, {len(TEXT.split())} words')
print(f'\nFirst 200 chars:\n{TEXT[:200]}...')

## 🔗 Part 1: Word-Level Markov Chain (From Scratch)
Each **state** = tuple of N consecutive words. The model records which words follow each state and picks randomly from observed transitions.

In [ ]:
class WordMarkovChain:
    def __init__(self, order=2):
        self.order = order
        self.chain = defaultdict(list)
        self.start_states = []

    def train(self, text):
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        for sentence in sentences:
            words = sentence.split()
            if len(words) <= self.order:
                continue
            self.start_states.append(tuple(words[:self.order]))
            for i in range(len(words) - self.order):
                state = tuple(words[i:i + self.order])
                self.chain[state].append(words[i + self.order])
        print(f'✅ Trained | order={self.order} | {len(self.chain)} states')

    def generate(self, seed=None, max_words=80, num_sequences=3):
        results = []
        for i in range(num_sequences):
            if seed:
                seed_words = seed.split()
                current = tuple(seed_words[-self.order:])
                words = list(seed_words)
            else:
                current = random.choice(self.start_states)
                words = list(current)
            for _ in range(max_words - self.order):
                next_words = self.chain.get(current)
                if not next_words:
                    break
                next_word = random.choice(next_words)
                words.append(next_word)
                current = tuple(words[-self.order:])
            text = ' '.join(words)
            results.append(text)
            print(f'\n[Generation {i+1}]\n{text}')
        return results

### Order 1 — Unigram Context

In [ ]:
wmc1 = WordMarkovChain(order=1)
wmc1.train(TEXT)
_ = wmc1.generate(max_words=60, num_sequences=2)

### Order 2 — Bigram Context

In [ ]:
wmc2 = WordMarkovChain(order=2)
wmc2.train(TEXT)
_ = wmc2.generate(max_words=80, num_sequences=2)

### Order 3 — Trigram Context (with seed)

In [ ]:
wmc3 = WordMarkovChain(order=3)
wmc3.train(TEXT)
_ = wmc3.generate(seed='Artificial intelligence is', max_words=80, num_sequences=2)

## 📊 Part 2: Transition Probability Analysis
Inspect which words most commonly follow each state.

In [ ]:
print('Top States and Transition Probabilities\n')
sorted_states = sorted(wmc2.chain.items(), key=lambda x: len(x[1]), reverse=True)
for state, transitions in sorted_states[:8]:
    total = len(transitions)
    freq = defaultdict(int)
    for w in transitions:
        freq[w] += 1
    top = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:3]
    probs = ', '.join(f"{w!r} ({c/total:.0%})" for w, c in top)
    print(f'State {state}  →  {probs}')

## 🔤 Part 3: Character-Level Markov Chain
Works at the character level instead of word level — can invent new words!

In [ ]:
class CharMarkovChain:
    def __init__(self, order=4):
        self.order = order
        self.chain = defaultdict(list)

    def train(self, text):
        for i in range(len(text) - self.order):
            state = text[i:i + self.order]
            self.chain[state].append(text[i + self.order])
        print(f'✅ Char chain trained | order={self.order} | {len(self.chain)} states')

    def generate(self, seed=None, max_chars=400, num_sequences=2):
        results = []
        all_states = list(self.chain.keys())
        for i in range(num_sequences):
            if seed and len(seed) >= self.order:
                current = seed[-self.order:]
                result = seed
            else:
                current = random.choice(all_states)
                result = current
            for _ in range(max_chars - self.order):
                next_chars = self.chain.get(current)
                if not next_chars:
                    current = random.choice(all_states)
                    continue
                next_char = random.choice(next_chars)
                result += next_char
                current = result[-self.order:]
            results.append(result)
            print(f'\n[Char Generation {i+1}]\n{result}')
        return results

cmc = CharMarkovChain(order=4)
cmc.train(TEXT)
_ = cmc.generate(max_chars=300, num_sequences=2)

## 📦 Part 4: markovify Library
Using the `markovify` library for clean, sentence-aware generation.

In [ ]:
model = markovify.Text(TEXT, state_size=2)
print('markovify | state_size=2\n')
for i in range(5):
    sentence = model.make_short_sentence(max_chars=280, tries=100)
    if sentence:
        print(f'[{i+1}] {sentence}')

## 🎛️ Part 5: Interactive Generation
Change the prompt and settings below and re-run!

In [ ]:
# ── Tweak these ──────────────────────────
YOUR_PROMPT = 'Machine learning is'   # Starting seed
ORDER       = 2                        # Chain order (1, 2, or 3)
MAX_WORDS   = 100                      # Max words to generate
N_OUTPUTS   = 3                        # Number of sequences
# ─────────────────────────────────────────

model_interactive = WordMarkovChain(order=ORDER)
model_interactive.train(TEXT)
_ = model_interactive.generate(
    seed=YOUR_PROMPT,
    max_words=MAX_WORDS,
    num_sequences=N_OUTPUTS,
)

## 🔑 Key Takeaways

| Order | Context | Output Quality |
|---|---|---|
| 1 | 1 word | Random, incoherent |
| 2 | 2 words | Readable phrases |
| 3 | 3 words | Coherent sentences |
| 4+ | 4+ words | Closely mirrors training data |

**Markov chains** are simple, interpretable, and require zero neural network training — making them a great introduction to probabilistic text generation before diving into transformers like GPT-2.